In [1]:
import os
import sys

from pyspark.sql import SparkSession


# ============================================================
# Python
# ============================================================

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


# ============================================================
# JARs
# ============================================================

HUDI_JAR = (
    r"C:\data\spark\jars"
    r"\hudi-spark4.0-bundle_2.13-1.2.0.jar"
)

ICEBERG_JAR = (
    r"C:\data\spark\jars"
    r"\iceberg-spark-runtime-4.0_2.13-1.10.0.jar"
)


# ============================================================
# MinIO locations
# ============================================================

RAW_PATH    = "s3a://rawload"
BRONZE_PATH = "s3a://bronzeload"
SILVER_PATH = "s3a://silverload"
GOLD_PATH   = "s3a://goldload"


# ============================================================
# Spark
# ============================================================

spark = (
    SparkSession.builder
    .appName("Local Lakehouse - MinIO")

    # --------------------------------------------------------
    # Hudi + Iceberg JARs
    # --------------------------------------------------------

    .config(
        "spark.jars",
        ",".join([
            HUDI_JAR,
            ICEBERG_JAR
        ])
    )

    # --------------------------------------------------------
    # Hudi
    # --------------------------------------------------------

    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    )

    .config(
        "spark.sql.extensions",
        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension,"
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )

    .config(
        "spark.kryo.registrator",
        "org.apache.spark.HoodieSparkKryoRegistrar"
    )

    # --------------------------------------------------------
    # Iceberg named catalog
    # --------------------------------------------------------

    .config(
        "spark.sql.catalog.ice",
        "org.apache.iceberg.spark.SparkCatalog"
    )

    .config(
        "spark.sql.catalog.ice.catalog-impl",
        "org.apache.iceberg.hadoop.HadoopCatalog"
    )

    .config(
        "spark.sql.catalog.ice.warehouse",
        SILVER_PATH
    )

    # --------------------------------------------------------
    # MinIO / S3A
    # --------------------------------------------------------

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )

    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )

    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )

    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    .getOrCreate()
)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [2]:
print("Spark:", spark.version)

print(
    "Serializer:",
    spark.sparkContext.getConf().get(
        "spark.serializer",
        "NOT SET"
    )
)

print(
    spark.sparkContext._jsc.sc().listJars()
)

NameError: name 'spark' is not defined